# Equality & Clustering Simulations (density held fixed)

Equality-variant of `2. GColab Simulations.ipynb`. Runs the Bayesian-agent bandit
simulations on the PUD and Tobacco citation networks under **four variation methods
that all hold network density constant**, so the manipulated dimensions are degree
equality and clustering — never density.

| Method | Mechanism | Invariant |
|---|---|---|
| `randomization` | rewire *k* random edges (remove one / add one) | `|E|` fixed |
| `equalize` | rewire *k* triangle edges toward degree equality | `|E|` fixed |
| `cluster` | degree-preserving double-edge swaps raising avg. clustering | `|E|` **and** degree sequence fixed |
| `decluster` | same swaps, lowering avg. clustering | `|E|` **and** degree sequence fixed |

`proportion_edges` (uniform on `[0, 1/3]`) is the intensity knob for all four: for
`randomization`/`equalize` it is the fraction of edges rewired; for
`cluster`/`decluster` it is the fractional shift in average clustering relative to
baseline.

The density arms (`densify`, `densify_fixed`) live in the original notebook and are
deliberately excluded here. The clustering arms use
`networks.variation_methods.generate_network_variant(..., n_edges=0)` rather than
`utils.network_utils.cluster_network`, because the latter is purely additive and
would confound clustering with density.

Outputs: `{pud,tobacco}_results_equality_{method}.csv` in the Drive dumping path.

# Setup

In [ ]:
import shutil, os
if os.path.exists('e_network_inequality'):
    shutil.rmtree('e_network_inequality')

# !git clone https://github.com/IgnacioOQ/e_network_inequality
!git clone -b main https://github.com/IgnacioOQ/e_network_inequality

In [ ]:
!pip install dill

In [ ]:
%cd e_network_inequality

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from utils.imports import *
from model.agents import BetaAgent, BayesAgent
from model.model import Model
from utils.network_utils import *
from networks.network_generation import *
from networks.variation_methods import *
from model.simulation_functions import *
from model.vectorized_simulation_functions import *
from functools import partial
import hashlib
import gc
from multiprocessing import get_context

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dumping_path = '/content/drive/My Drive/Colab Projects/Data Driven ABMs/Data Sets/equality_study/'
print("Current Directory:", dumping_path)

In [ ]:
# ══ Study parameters — every tunable knob for this notebook lives here ══
from numpy.random import SeedSequence

n_simulations = 10000

# Equality / clustering arms only. Every method below holds |E| fixed, so
# density is constant across the whole study; 'densify' / 'densify_fixed'
# are deliberately excluded (see the header cell).
methods = ['randomization', 'equalize', 'cluster', 'decluster']

# Pool size. cpu_count() is often too aggressive on a shared cloud runtime —
# each worker holds its own copy of the network plus the simulation state, so
# memory, not CPU, is usually the binding constraint. Raise MAX_CORES if the
# runtime has headroom.
MAX_CORES = 8
num_cores = min(cpu_count(), MAX_CORES)
print(f"cpu_count()={cpu_count()} -> using num_cores={num_cores}")

# One master seed per network, so PUD and Tobacco never share random streams.
# Same seed + same method order reproduces the study exactly; per-run seeds are
# also written into result_dict by run_vectorized_simulation_with_params.
MASTER_SEEDS = {'pud': 20260508, 'tobacco': 20260509}

# Stopping rule for every simulation in this study.
SIM_KWARGS = dict(
    tolerance=5e-3,
    tolerance_stopping=False,
    tstep_stopping=True,
    number_of_steps=10000,
)

print(f"methods={methods}\nn_simulations={n_simulations} per method per network")

In [ ]:
def generate_parameters_here(_,G,method='randomization'):
    # ── Equality / clustering study — density is held fixed in EVERY branch ──
    # 'randomization', 'equalize'            : remove-one/add-one  -> |E| unchanged
    # 'cluster', 'decluster'                 : degree-preserving double-edge swaps
    #                                          (generate_network_variant with
    #                                           n_edges=0) -> |E| AND the full
    #                                           degree sequence unchanged
    #
    # NOTE: the older utils.network_utils.cluster_network is deliberately NOT used
    # here — it is purely additive (adds n edges, removes none), which would
    # confound clustering with density. The density arms ('densify',
    # 'densify_fixed') live in "2. GColab Simulations.ipynb".
    process_seed = int.from_bytes(os.urandom(4), byteorder='little')
    rd.seed(process_seed)
    # Seed stdlib `random` too, not just numpy.random (`rd`). The variation
    # helpers (equalize, generate_network_variant) draw from stdlib `random`.
    # CPython auto-reseeds it per forked child, so variance was never at risk;
    # this is for REPRODUCIBILITY -- process_seed is logged as
    # result['parameter_random_seed'], letting a run's variation be replayed.
    random.seed(process_seed)
    # Randomly sample parameters for this group
    uncertainty = rd.uniform(.000001, .001)
    n_experiments = rd.randint(1000, 10000)
    # now we pick a random number
    # Capped at 1/3 to prevent "Sample larger than population" errors in equalize
    proportion_edges = rd.rand() * 0.1
    # Do randomization
    num_edges = G.number_of_edges()
    if method == 'randomization':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = randomize_network(G, n_edges=num_edges_to_randomize)
    if method == 'equalize':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = generate_equalize_variant(G, n_edges=num_edges_to_randomize)[0]
    if method in ('cluster', 'decluster'):
      # proportion_edges doubles as the clustering intensity: move average
      # clustering up (cluster) or down (decluster) by up to 1/3 of baseline.
      base_clustering = float(np.average(list(nx.clustering(G).values())))
      sign = 1.0 if method == 'cluster' else -1.0
      target_clustering = base_clustering * (1.0 + sign * proportion_edges)
      modified_network = generate_network_variant(
          G,
          n_edges=0,                       # add no edges -> density fixed
          target_clustering=target_clustering,
          max_post_rewires=10 * num_edges, # generous cap; stops early on tolerance
          rewiring_tolerance=1e-3,
      )[0]

    result = generate_parameters_aggregate(modified_network, uncertainty=uncertainty, n_experiments=n_experiments,
                                           p_rewiring=proportion_edges)
    result['uncertainty'] = uncertainty
    result['n_experiments'] = n_experiments
    result['proportion_edges'] = proportion_edges
    result['parameter_random_seed']= process_seed
    return result

In [ ]:
def run_method(method, G, master_ss, prefix):
    """Run `n_simulations` simulations for one variation method on network `G`.

    Defined ONCE and reused by both the PUD and Tobacco sections — `G`,
    `master_ss` and `prefix` are explicit parameters rather than globals, so
    there is no hidden state and no silent redefinition between sections.

    Parameters
    ----------
    method : str      one of `methods` — the network variation arm
    G : nx.DiGraph    the (index-relabelled) base network
    master_ss : SeedSequence  per-network master; child seeds spawn from it
    prefix : str      output filename prefix, e.g. 'pud' or 'tobacco'

    Writes/appends `{prefix}_results_equality_{method}.csv` in `dumping_path`.
    """
    print(f'Running stuff for method: {method}')

    # ─── Generate parameters ───
    print('Generating parameters...')
    generate_params = partial(generate_parameters_here, G=G, method=method)

    with Pool(num_cores) as pool:
        param_dict = list(tqdm(
            pool.imap_unordered(generate_params, range(n_simulations)),
            total=n_simulations
        ))

    # ─── Assign per-job simulation seeds ───
    child_seeds = [int(s.generate_state(1)[0])
                   for s in master_ss.spawn(len(param_dict))]
    for pd_, cs in zip(param_dict, child_seeds):
        pd_["seed"] = cs

    # ─── Run simulations ───
    run_simulation_wrapper = partial(run_vectorized_simulation_with_params,
                                     **SIM_KWARGS)
    with Pool(num_cores) as pool:
        simulation_results = list(tqdm(
            pool.imap_unordered(run_simulation_wrapper, param_dict),
            total=len(param_dict),
            desc="Running simulations"
        ))

    # ─── Save results ───
    results_path = dumping_path + f"{prefix}_results_equality_{method}.csv"
    new_results_df = pd.DataFrame(simulation_results)

    if os.path.exists(results_path):
        existing_results_df = pd.read_csv(results_path)
        combined_results_df = pd.concat([existing_results_df, new_results_df], ignore_index=True)
    else:
        combined_results_df = new_results_df

    combined_results_df.to_csv(results_path, index=False)
    print(len(combined_results_df), "\n")

# Testing Peptic Ulcer

In [ ]:
with open('./networks/citation_data/pud_network.pkl', 'rb') as f:
  G_pud = pickle.load(f)

# Relabel nodes to contiguous integer indices (the vectorized model indexes the
# adjacency matrix positionally).
mapping = {node: index for index, node in enumerate(G_pud.nodes())}
G_pud_indexed = nx.relabel_nodes(G_pud, mapping)

# Sanity-check the loaded network before spending compute on it.
assert G_pud_indexed.number_of_nodes() == G_pud.number_of_nodes()
assert G_pud_indexed.number_of_edges() == G_pud.number_of_edges()
assert not list(nx.selfloop_edges(G_pud_indexed)), "self-loops present"

print(f"PUD: {G_pud_indexed.number_of_nodes()} nodes, "
      f"{G_pud_indexed.number_of_edges()} edges, "
      f"avg clustering {nx.average_clustering(G_pud_indexed):.5f}")

In [ ]:
%%time
# Each call to run_method spawns a fresh sub-sequence from master_ss, so seeds
# across (method, run) pairs are statistically independent.
master_ss = SeedSequence(MASTER_SEEDS['pud'])

for method in methods:
    run_method(method, G=G_pud_indexed, master_ss=master_ss, prefix='pud')

## Basic Plotting Peptic Ulcer

In [ ]:
for method in methods:
  print(f'Plots for method: {method}')
  results_path = dumping_path + "pud_results_equality_"+method+".csv"
  combined_results_df = pd.read_csv(results_path)
  print(len(combined_results_df))
  scatter_plot(combined_results_df)
  scatter_plot(combined_results_df, target_variable="convergence_step")
  print('\n')

In [ ]:
# scatter_plot(combined_results_df)

In [ ]:
# scatter_plot(combined_results_df, target_variable="convergence_step")

# Testing Tobacco

In [ ]:
with open('./networks/citation_data/tobacco_network.pkl', 'rb') as f:
  G_tobacco = pickle.load(f)

# Relabel nodes to contiguous integer indices (see the PUD cell).
mapping = {node: index for index, node in enumerate(G_tobacco.nodes())}
G_tobacco_indexed = nx.relabel_nodes(G_tobacco, mapping)

# Sanity-check the loaded network before spending compute on it.
assert G_tobacco_indexed.number_of_nodes() == G_tobacco.number_of_nodes()
assert G_tobacco_indexed.number_of_edges() == G_tobacco.number_of_edges()
assert not list(nx.selfloop_edges(G_tobacco_indexed)), "self-loops present"

print(f"Tobacco: {G_tobacco_indexed.number_of_nodes()} nodes, "
      f"{G_tobacco_indexed.number_of_edges()} edges, "
      f"avg clustering {nx.average_clustering(G_tobacco_indexed):.5f}")

In [ ]:
%%time
# Distinct master seed from PUD, so the two networks never share streams.
master_ss = SeedSequence(MASTER_SEEDS['tobacco'])

for method in methods:
    run_method(method, G=G_tobacco_indexed, master_ss=master_ss, prefix='tobacco')

## Basic Plotting Tobacco

In [ ]:
for method in methods:
  print(f'Plots for method: {method}')
  results_path = dumping_path + "tobacco_results_equality_"+method+".csv"
  combined_results_df = pd.read_csv(results_path)
  print(len(combined_results_df))
  scatter_plot(combined_results_df)
  scatter_plot(combined_results_df, target_variable="convergence_step")
  print('\n')

## Disconnect from Runtime

In [ ]:
from datetime import datetime
import pytz
from IPython.display import Javascript

# Get current time in New York
nyc_time = datetime.now(pytz.timezone('America/New_York'))
formatted_time = nyc_time.strftime('%Y-%m-%d %H:%M:%S %Z')

# Print and log
print(f"✅ Disconnected from runtime at: {formatted_time}")

# Disconnect Colab runtime
display(Javascript('google.colab.kernel.disconnect()'))